# Hardread TCG — BC Quality Cleaning
Kaggle-only Phase 2B. It creates weighted BC datasets; it does not train or submit.

In [ ]:
RUN_PHASE_QUALITY = True
MAX_FILES = 5_000
MAX_PAIRS = 250_000
INPUT_ROOT = '/kaggle/input'
OUT_ROOT = '/kaggle/working/bc_quality_v1'
print({'quality':RUN_PHASE_QUALITY,'max_files':MAX_FILES,'max_pairs':MAX_PAIRS})


In [ ]:
import os, sys, runpy
SRC=None
for root, dirs, files in os.walk(INPUT_ROOT):
    if 'featurizer.py' in files and 'replay.py' in files:
        SRC=root;break
if SRC is None: raise FileNotFoundError('Attach hardread-tcg-src')
sys.path.insert(0, os.path.dirname(SRC));sys.path.insert(0,SRC)
QUALITY_SCRIPT='#!/usr/bin/env python3\n"""Kaggle-only BC-quality dataset builder; never writes raw data locally."""\nfrom __future__ import annotations\nimport argparse, collections, glob, hashlib, json, math, os, time\nfrom pathlib import Path\nimport numpy as np\nfrom src.replay import parse_episode, _collect_player_card_ids, match_deck, featurize_pair, _flush_shard\nfrom src.featurizer import MAX_OPTIONS\n\ndef reward(x): return float(x) if isinstance(x,(int,float,np.integer,np.floating)) else 0.0\ndef wilson_lb(w,n,z=1.96):\n    if n<=0:return 0.0\n    p=w/n; den=1+z*z/n\n    return max(0.0,(p+z*z/(2*n)-z*math.sqrt((p*(1-p)+z*z/(4*n))/n))/den)\ndef raw_files(root,max_files):\n    out=[]\n    for p in glob.glob(os.path.join(root,\'**\',\'*.json\'),recursive=True):\n        try:\n            with open(p) as f:x=json.load(f)\n            if isinstance(x,dict) and isinstance(x.get(\'steps\'),list) and isinstance(x.get(\'info\'),dict):out.append(p)\n        except Exception:pass\n        if max_files and len(out)>=max_files:break\n    return sorted(out)\ndef scan(files):\n    agents=collections.defaultdict(collections.Counter); eps=[]; seen=set(); duplicates=0\n    for fp in files:\n        e=parse_episode(fp)\n        if not e:continue\n        info=e.get(\'info\') or {}; eid=info.get(\'EpisodeId\')\n        if eid in seen:duplicates+=1;continue\n        seen.add(eid); names=(info.get(\'TeamNames\') or [])[:2]; rs=e.get(\'rewards\') or [0,0];r0,r1=reward(rs[0] if rs else 0),reward(rs[1] if len(rs)>1 else 0)\n        winner=0 if r0>r1 else 1 if r1>r0 else -1\n        statuses=(e.get(\'statuses\') or [])[:2]\n        for p,n in enumerate(names):\n            if not n:continue\n            if winner<0:agents[n][\'draw\']+=1\n            elif p==winner:agents[n][\'win\']+=1\n            else:agents[n][\'loss\']+=1\n        eps.append((fp,eid,names,winner,statuses))\n    stats={n:{\'wins\':s[\'win\'],\'losses\':s[\'loss\'],\'draws\':s[\'draw\'],\'games\':s[\'win\']+s[\'loss\'],\'wilson_lcb\':wilson_lb(s[\'win\'],s[\'win\']+s[\'loss\'])} for n,s in agents.items()}\n    eligible=[n for n,x in stats.items() if x[\'games\']>=30]\n    eligible.sort(key=lambda n:(stats[n][\'wilson_lcb\'],stats[n][\'games\']),reverse=True)\n    top=set(eligible[:max(1,int(.30*len(eligible)))])\n    return eps,stats,top,duplicates\ndef valid_entries(ep, allowed, stats, variant, min_decisions=8, cap=96):\n    info=ep.get(\'info\') or {}; names=(info.get(\'TeamNames\') or [])[:2];rs=ep.get(\'rewards\') or [0,0];r0,r1=reward(rs[0] if rs else 0),reward(rs[1] if len(rs)>1 else 0)\n    winner=0 if r0>r1 else 1 if r1>r0 else -1\n    statuses=(ep.get(\'statuses\') or [])[:2]\n    if winner<0 or len(statuses)<2 or any(s!=\'DONE\' for s in statuses):return [],{\'terminal_or_draw\':1}\n    did={p:match_deck(_collect_player_card_ids(ep.get(\'steps\') or [],p)) for p in(0,1)}\n    candidates=[[],[]];rej=collections.Counter()\n    for step in ep.get(\'steps\') or []:\n      if not isinstance(step,list):continue\n      for p,e in enumerate(step[:2]):\n        if not isinstance(e,dict) or (variant==\'top\' and (p>=len(names) or names[p] not in allowed)):continue\n        obs=e.get(\'observation\');a=e.get(\'action\');sel=(obs or {}).get(\'select\') if isinstance(obs,dict) else None\n        if not isinstance(sel,dict) or not sel.get(\'option\') or not isinstance(a,list) or not a:continue\n        opts=sel.get(\'option\') or []; mc=int(sel.get(\'maxCount\',0));mn=int(sel.get(\'minCount\',0))\n        if int(sel.get(\'type\',-1))==9 or len(a)>=30:rej[\'deck_selection\']+=1;continue\n        if any(not isinstance(i,(int,np.integer)) or not 0<=int(i)<len(opts) for i in a):rej[\'illegal_index\']+=1;continue\n        if len(set(map(int,a)))!=len(a):rej[\'duplicate_action\']+=1;continue\n        if not mn<=len(a)<=mc:rej[\'count_violation\']+=1;continue\n        candidates[p].append((obs,[int(i) for i in a],winner,did[p],len(opts)==1))\n    if sum(map(len,candidates))<min_decisions:return [],rej|{\'too_short\':1}\n    out=[]\n    for p,rows in enumerate(candidates):\n      # deterministic even subsample to prevent a long episode dominating training.\n      if len(rows)>cap: rows=[rows[i] for i in np.linspace(0,len(rows)-1,cap,dtype=int)];rej[\'episode_cap\']+=1\n      n=names[p] if p<len(names) else \'\'; st=stats.get(n,{}); strength=float(st.get(\'wilson_lcb\',0.0)); agent_w=max(.20,min(1.0,.30+1.4*strength))\n      for obs,a,w,d,forced in rows:\n        outcome=1.0 if p==w else -1.0\n        rec=featurize_pair(obs,a,deck_id=d,outcome=outcome,episode_id=int(info.get(\'EpisodeId\') or 0))\n        if rec is None:rej[\'featurize\']+=1;continue\n        # `legal_mask` deliberately uses -inf for padded slots; audit only\n        # numeric model inputs that must be finite.\n        finite_keys=(\'board\',\'options\',\'card_ids\',\'option_card\',\'max_count\',\'min_count\',\'select_type\',\'select_ctx\',\'deck_id\',\'chosen\',\'outcome\',\'episode_id\')\n        if not all(np.isfinite(rec[k]).all() for k in finite_keys):rej[\'nonfinite\']+=1;continue\n        order=np.full(MAX_OPTIONS,-1,dtype=np.int32);order[:len(a)]=a\n        rec[\'chosen_order\']=order;rec[\'action_len\']=np.int32(len(a));rec[\'sample_weight\']=np.float32(agent_w*(1.0 if outcome>0 else .35)*(.05 if forced else 1.0)*(.5 if d==3 else 1.0));rec[\'agent_strength\']=np.float32(strength);rec[\'forced_action\']=np.int8(forced)\n        out.append(rec)\n    return out,rej\ndef digest(r):\n    h=hashlib.sha1()\n    for k in (\'board\',\'card_ids\',\'options\',\'option_card\',\'legal_mask\',\'chosen_order\'):h.update(np.ascontiguousarray(r[k]).tobytes())\n    return h.digest()\ndef build(eps,stats,top,out_dir,variant,limit):\n    Path(out_dir).mkdir(parents=True,exist_ok=True); shard=[];seen=set(); q=collections.Counter(); deck=collections.Counter(); total=0;idx=0;used=0\n    for fp,eid,names,winner,statuses in eps:\n      ep=parse_episode(fp)\n      rows,rej=valid_entries(ep,top,stats,variant);q.update(rej)\n      for r in rows:\n        d=digest(r)\n        if d in seen:q[\'exact_duplicate\']+=1;continue\n        seen.add(d);shard.append(r);total+=1;deck[int(r[\'deck_id\'])]+=1\n        if len(shard)>=5000:_flush_shard(shard,out_dir,idx);idx+=1;shard=[]\n        if limit and total>=limit:break\n      if rows:used+=1\n      if limit and total>=limit:break\n    if shard:_flush_shard(shard,out_dir,idx);idx+=1\n    return {\'pairs\':total,\'shards\':idx,\'episodes\':used,\'deck_pairs\':dict(deck),\'rejections\':dict(q)}\ndef integrity(root):\n n=bad=0\n for f in glob.glob(os.path.join(root,\'shard_*.npz\')):\n  z=np.load(f);n+=len(z[\'chosen\']);bad+=int(((z[\'chosen\']>0).sum(1)>z[\'max_count\']).sum());bad+=int(((z[\'chosen_order\']>=0).sum(1)!=z[\'action_len\']).sum())\n return {\'pairs\':n,\'violations\':bad}\ndef main():\n ap=argparse.ArgumentParser();ap.add_argument(\'--input-root\',default=\'/kaggle/input\');ap.add_argument(\'--out\',default=\'/kaggle/working/bc_quality_v1\');ap.add_argument(\'--max-files\',type=int,default=5000);ap.add_argument(\'--max-pairs\',type=int,default=250000);a=ap.parse_args();out=Path(a.out);out.mkdir(parents=True,exist_ok=True)\n files=raw_files(a.input_root,a.max_files);eps,stats,top,dups=scan(files)\n (out/\'agent_quality.json\').write_text(json.dumps({\'agents\':stats,\'top_agents\':sorted(top),\'duplicate_episode_files\':dups},indent=2))\n result={\'source_files\':len(files),\'unique_episodes\':len(eps),\'weighted_all\':build(eps,stats,top,str(out/\'weighted_all\'),\'all\',a.max_pairs),\'weighted_top\':build(eps,stats,top,str(out/\'weighted_top\'),\'top\',a.max_pairs)}\n result[\'integrity\']={\'weighted_all\':integrity(str(out/\'weighted_all\')),\'weighted_top\':integrity(str(out/\'weighted_top\'))};(out/\'summary.json\').write_text(json.dumps(result,indent=2));print(json.dumps(result,indent=2))\nif __name__==\'__main__\':main()\n'
PATH='/kaggle/working/kaggle_build_bc_quality.py'
open(PATH,'w').write(QUALITY_SCRIPT)
print('source',SRC)


## Cleaning policy
Hard rejects: non-DONE/drawn games, deck selection, illegal/out-of-range/duplicate actions, count violations, nonfinite features, duplicate episode IDs and exact duplicate examples.

Weights: Wilson 95%% lower-bound agent strength (minimum 30 games), winner=1, loser=.35, forced=.05, unknown deck=.5. Episodes with fewer than 8 decisions are dropped; each player contributes at most 96 decisions.

In [ ]:
if RUN_PHASE_QUALITY:
    sys.argv=['kaggle_build_bc_quality.py','--input-root',INPUT_ROOT,'--out',OUT_ROOT,'--max-files',str(MAX_FILES),'--max-pairs',str(MAX_PAIRS)]
    runpy.run_path(PATH,run_name='__main__')


Outputs stay in Kaggle:
- `bc_quality_v1/weighted_all/`
- `bc_quality_v1/weighted_top/`
- `bc_quality_v1/agent_quality.json`
- `bc_quality_v1/summary.json`
No training is enabled in this notebook.

In [ ]:
import glob
print(sorted(glob.glob(OUT_ROOT+'/**',recursive=True))[:20])
